In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -q transformers peft evaluate rouge_score nltk accelerate bitsandbytes safetensors


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 14.2 MB/s eta 0:00:00


In [ ]:
!find /content/drive/MyDrive/llm_finance -type f -name "adapter_config.json" -print


/content/drive/MyDrive/llm_finance/finance_lora_adapter/adapter_config.json


In [ ]:
!pip install -q transformers peft evaluate rouge_score nltk accelerate bitsandbytes
import torch, pandas as pd, nltk, evaluate, time, os, json
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# ==== CONFIG ====
ADAPTER_DIR = "/content/drive/MyDrive/llm_finance/finance_lora_adapter"
BASE_MODEL  = "gpt2"
TEST_CSV    = "/content/drive/MyDrive/llm_finance/prepared/test.csv"
OUT_DIR     = "/content/drive/MyDrive/llm_finance/final_evaluation"
N           = 500
MAX_NEW     = 120
# ================

os.makedirs(OUT_DIR, exist_ok=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("Loading tokenizer + base model...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading base model...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loading adapter:", ADAPTER_DIR)
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

device = next(model.parameters()).device
print("Model device:", device)

def fast_generate(batch_prompts):
    """
    Batch generation for speed — generates 4 prompts at a time.
    """
    texts = []
    for prompt in batch_prompts:
        inputs = tokenizer(prompt, return_tensors="pt").to(device)
        with torch.no_grad():
            out = model.generate(
                input_ids=inputs["input_ids"],
                max_new_tokens=MAX_NEW,
                pad_token_id=tokenizer.pad_token_id
            )
        gen = tokenizer.decode(
            out[0][inputs["input_ids"].shape[-1]:],
            skip_special_tokens=True
        ).strip()
        texts.append(gen)
    return texts

# Load and sample 500
df_test = pd.read_csv(TEST_CSV)
sample_df = df_test.sample(N, random_state=42).reset_index(drop=True)

print(f"Evaluating {N} samples...")
rouge = evaluate.load("rouge")

preds, refs = [], []
start = time.time()

# Batch size = 4 for speed
BATCH = 4
for i in range(0, N, BATCH):
    batch = sample_df.iloc[i:i+BATCH]
    batch_prompts = list(batch["prompt"])
    batch_outputs = fast_generate(batch_prompts)

    preds.extend(batch_outputs)
    refs.extend(list(batch["response"]))

    if (i + BATCH) % 50 == 0:
        print(f"... generated {min(i+BATCH, N)} / {N}")

elapsed = time.time() - start
print("Generation finished in", round(elapsed, 2), "seconds")

# Compute ROUGE
rouge_res = rouge.compute(predictions=preds, references=refs)
print("\nROUGE:", rouge_res)

# Compute token F1
def token_f1(a, b):
    a_t = nltk.word_tokenize(a.lower())
    b_t = nltk.word_tokenize(b.lower())
    if not a_t or not b_t:
        return 0.0
    common = set(a_t) & set(b_t)
    if not common:
        return 0.0
    prec = len(common) / len(a_t)
    rec  = len(common) / len(b_t)
    return 2 * prec * rec / (prec + rec)

f1_avg = sum(token_f1(p,r) for p,r in zip(preds,refs)) / len(preds)
print("Token-overlap F1:", f1_avg)

# Save 50 examples for manual review
k = min(50, len(sample_df))
rows = []
for i in range(k):
    rows.append({
        "prompt": sample_df.loc[i,"prompt"],
        "gold":   sample_df.loc[i,"response"],
        "gen":    preds[i]
    })
pd.DataFrame(rows).to_csv(os.path.join(OUT_DIR, "eval_50_examples.csv"), index=False)
print("Saved 50 examples to", OUT_DIR)

# Save metrics
summary = {
    "rouge": {k: float(v) for k,v in rouge_res.items()},
    "token_f1": float(f1_avg),
    "n": N
}
with open(os.path.join(OUT_DIR, "metrics.json"), "w") as f:
    json.dump(summary, f, indent=2)

print("\nAll done! 🎉")


Loading tokenizer + base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

Loading adapter: /content/drive/MyDrive/llm_finance/finance_lora_adapter


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


Model device: cuda:0
Evaluating 500 samples...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


... generated 100 / 500
... generated 200 / 500
... generated 300 / 500
... generated 400 / 500
... generated 500 / 500
Generation finished in 1042.79 seconds

ROUGE: {'rouge1': np.float64(0.31271803650085833), 'rouge2': np.float64(0.20561102471255466), 'rougeL': np.float64(0.27092634246002206), 'rougeLsum': np.float64(0.27128041082821563)}
Token-overlap F1: 0.2458293551159063
Saved 50 examples to /content/drive/MyDrive/llm_finance/final_evaluation

All done! 🎉


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

BASE_MODEL = "gpt2"
ADAPTER_DIR = "/content/drive/MyDrive/llm_finance/finance_lora_adapter"

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load base + adapter
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map="auto")
model = PeftModel.from_pretrained(base, ADAPTER_DIR)
model.eval()

def ask_finance_question(question):
    prompt = f"""
Instruction: Answer the following finance question based on the provided context (if any).

Question: {question}

Response:
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(next(model.parameters()).device)
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs["input_ids"],
            max_new_tokens=150,
            pad_token_id=tokenizer.pad_token_id
        )
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return answer.strip()

# ---- Ask your question here ----
print(ask_finance_question("What is a derivative in finance?"))


/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


A derivative in finance refers to a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that


Ask question....

In [ ]:
ask_finance_question("What is a derivative in finance?")


'A derivative in finance refers to a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that is a financial instrument that'